# Load an Existing Chroma Database

This notebook reconnects to the persisted Chroma collection created in the CRUD notebook and reads its contents.

In [2]:
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_ollama.embeddings import OllamaEmbeddings

## 1. Rebuild the Same Configuration

In [3]:
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

project_root

PosixPath('/Users/itaakash/Desktop/Lakhan/Tasks/ragAdancedCampusX/Advanced_Rag_Codes-main/04_vector_stores')

In [4]:
# dotenv_path = project_root / ".env"
# load_dotenv(dotenv_path=dotenv_path)

# if not os.getenv("OPENAI_API_KEY"):
#     raise ValueError("Please add your OPENAI_API_KEY to the .env file before running this notebook.")

# print(f"Loaded environment from: {dotenv_path}")

In [5]:
collection_name = "demo_2"
persist_directory = project_root / "notebooks" / "chroma_langchain_db"

print(f"Collection name: {collection_name}")
print(f"Persist directory: {persist_directory}")

Collection name: demo_2
Persist directory: /Users/itaakash/Desktop/Lakhan/Tasks/ragAdancedCampusX/Advanced_Rag_Codes-main/04_vector_stores/notebooks/chroma_langchain_db


In [6]:
# Use the same embedding model that was used to build the store.
embeddings = OllamaEmbeddings(model="embeddinggemma")

vector_store = Chroma(
    collection_name=collection_name,
    embedding_function=embeddings,
    persist_directory=str(persist_directory),
)

print("Connected to the existing Chroma collection.")

Connected to the existing Chroma collection.


## 2. Add Small Display Helpers

In [7]:
def preview_text(text, limit=80):
    """Return a short preview for cleaner notebook output."""
    if len(text) <= limit:
        return text
    return text[:limit] + "..."


def print_stored_documents(records):
    """Print stored Chroma records in a readable format."""
    ids = records.get("ids", [])
    documents = records.get("documents", [])
    metadatas = records.get("metadatas", [])

    print(f"Total documents in collection: {len(ids)}")
    print()

    for index, (doc_id, document_text, metadata) in enumerate(zip(ids, documents, metadatas), start=1):
        print(f"{index}. id={doc_id}")
        print(f"   topic={metadata.get('topic')} | doc_number={metadata.get('doc_number')}")
        print(f"   content={preview_text(document_text)}")
        print()

## 3. Fetch and Inspect the Stored Records

In [8]:
stored_records = vector_store.get(include=["embeddings", "metadatas", "documents"])
stored_records.keys()

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas'])

In [9]:
stored_records["embeddings"].shape

(8, 768)

In [10]:
print_stored_documents(stored_records)

Total documents in collection: 8

1. id=8cdb3a2e-2bb4-43ae-8efe-0faa846a7211
   topic=AI | doc_number=1
   content=Artificial intelligence helps machines perform tasks that usually need human rea...

2. id=03cfc174-e179-4be0-ab37-ac473cf5d973
   topic=AI | doc_number=2
   content=AI systems can analyze patterns in data to support predictions and automation.

3. id=6fd46545-28fc-4a3d-bd97-e69e351e4202
   topic=AI | doc_number=3
   content=Responsible AI development includes fairness, transparency, and safety checks.

4. id=4e8de225-921f-4ee3-82d5-721f41f8cd8d
   topic=RAG | doc_number=4
   content=RAG improves answer quality by retrieving relevant context before the language m...

5. id=2f184dfb-bf53-403d-845f-4f6241f19147
   topic=RAG | doc_number=5
   content=A retriever in a RAG pipeline finds relevant chunks before the language model ge...

6. id=183020ce-f764-46c2-afc8-8bccf31ce62f
   topic=RAG | doc_number=6
   content=Vector stores are important in RAG because they make semantic 